In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

import numpy

import pandas as pd

import matplotlib.pyplot as plt

In [ ]:
x = pd.read_csv('../data/raw/bryant/allseqs_20191230.csv')
print(len(x))

In [5]:
x['is_viable'].value_counts()

is_viable
True     153692
False    143278
Name: count, dtype: int64

In [8]:
total = len(x)
unique = x['sequence'].nunique()
print(f"Total rows:       {total}")
print(f"Unique sequences: {unique}")
print(f"All unique:       {total == unique}")


Total rows:       296970
Unique sequences: 293574
All unique:       False


In [9]:
# How many rows each unique sequence appears across partitions
seq_counts = x.groupby('sequence')['partition'].value_counts().rename('count').reset_index()
print("Unique sequences per partition:")
print(x.groupby('partition')['sequence'].nunique().sort_values(ascending=False))
print()

# Sequences that appear in multiple partitions
multi = x.groupby('sequence')['partition'].nunique()
print(f"\nSequences in >1 partition: {(multi > 1).sum()}")
print(multi[multi > 1].value_counts())

# Overall partition distribution
print("\nRow counts per partition:")
print(x['partition'].value_counts())


Unique sequences per partition:
partition
designed                                56372
random_doubles                          25040
rnn_standard_walked                     20838
cnn_designed_plus_rand_train_walked     20759
rnn_designed_plus_rand_train_walked     20731
lr_standard_walked                      20456
cnn_rand_doubles_plus_single_walked     20454
cnn_standard_walked                     20395
rnn_rand_doubles_plus_singles_walked    20154
lr_rand_doubles_plus_single_walked      19999
lr_designed_plus_rand_train_walked      19680
rand                                     9885
lr_rand_doubles_plus_single_seed         2071
rnn_designed_plus_rand_train_seed        2065
rnn_rand_doubles_plus_singles_seed       2045
lr_designed_plus_rand_train_seed         2030
cnn_rand_doubles_plus_single_seed        2022
lr_standard_seed                         1989
cnn_standard_seed                        1924
rnn_standard_seed                        1916
cnn_designed_plus_rand_train_seed     

In [10]:
dupe_seqs = x[x.duplicated('sequence', keep=False)].sort_values('sequence')
print(f"Rows with duplicate sequences: {len(dupe_seqs)}")
dupe_seqs[['sequence', 'partition', 'is_viable']].to_string(index=False)


Rows with duplicate sequences: 6792


'                                  sequence               partition  is_viable\n            ADTIIVTTNPVATEQYGCVgAeMNLQPLIR                designed      False\n            ADTIIVTTNPVATEQYGCVgAeMNLQPLIR previous_chip_nonviable      False\n              AEAEISTTNPVATECYGNVGTNLQRGTQ                designed       True\n              AEAEISTTNPVATECYGNVGTNLQRGTQ    previous_chip_viable       True\n          AEDEVKGTNPVSTEPYGQCSaDLLQnIaGpQY                designed      False\n          AEDEVKGTNPVSTEPYGQCSaDLLQnIaGpQY previous_chip_nonviable      False\n              AEDEVSTTNPVCTEMWGFISENLQHFNR                designed      False\n              AEDEVSTTNPVCTEMWGFISENLQHFNR previous_chip_nonviable      False\n              AEEEIAQTNPEATEQYGDVSTNLQRGER                designed       True\n              AEEEIAQTNPEATEQYGDVSTNLQRGER    previous_chip_viable       True\n              AEEEIAQTNPFSTEQYGVVSEDLEVGED                designed      False\n              AEEEIAQTNPFSTEQYGVVSEDLEVGED previous

In [ ]:

#quantify which partition co-occurrence patterns drive the duplicates
pair_counts = {}
for seq, group in x[x.duplicated('sequence', keep=False)].groupby('sequence'):
    partitions = tuple(sorted(group['partition'].unique()))
    pair_counts[partitions] = pair_counts.get(partitions, 0) + 1

pair_df = pd.Series(pair_counts).sort_values(ascending=False)
print(f"Distinct co-occurrence patterns: {len(pair_df)}\n")
print(pair_df.head(20).to_string())


Distinct co-occurrence patterns: 9

single                   singles                    1085
designed                 previous_chip_viable        949
                         previous_chip_nonviable     723
previous_chip_nonviable  rand                        266
designed                 random_doubles              248
rand                     random_doubles               71
previous_chip_viable     rand                         27
                         single                       18
previous_chip_nonviable  single                        9


In [6]:
random_partitions = ['rand', 'random_doubles', 'single', 'singles']

groups = {
    'designed':      x[x['partition'] == 'designed']['is_viable'],
    'rand':          x[x['partition'] == 'rand']['is_viable'],
    'random_doubles':x[x['partition'] == 'random_doubles']['is_viable'],
    'single':        x[x['partition'] == 'single']['is_viable'],
    'singles':       x[x['partition'] == 'singles']['is_viable'],
    'random (all)':  x[x['partition'].isin(random_partitions)]['is_viable'],
}

for name, col in groups.items():
    pct = col.mean() * 100
    print(f"{name:<20} {pct:.1f}%  (n={len(col):,})")


designed             62.5%  (n=56,372)
rand                 9.8%  (n=9,885)
random_doubles       18.5%  (n=25,040)
single               58.1%  (n=1,112)
singles              38.3%  (n=1,085)
random (all)         17.9%  (n=37,122)


In [2]:
import torch
seqs = torch.load('../data/processed/bryant/scheme_a/diffusion_seq.pt')

/var/folders/m3/ydq73xbd307f31yp54h2vqhm0000gn/T/ipykernel_17096/2029933866.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  seqs = torch.load('../data/processed/bryant/s

In [8]:
lenghts = []
for i in seqs:
    lenghts.append(len(i))
import numpy as np
print(np.unique(lenghts, return_counts=True))

(array([28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43]), array([33926, 22260, 19915, 15607, 15214, 15283, 15747, 15235, 16283,
       16488, 16229, 14429,  8954,  5231,  2640,  1313]))
